# 2.1 Choosing the modeling approach

**Story:** SafeDrive has clean claims data and policy documents. Before building anything, you decide
*what kind* of model each problem needs: a classic ML algorithm, a ready-made AWS AI service, or a
foundation model (FM) on Amazon Bedrock.

**Task 2.1:** Choose appropriate modeling approaches for ML and AI solutions.

| Skill | Topic | Status |
|---|---|---|
| 2.1.1 | Select Bedrock FMs for a task | ✅ |
| 2.1.2 | Fine-tuning strategies for FMs | 📝 |
| 2.1.3 | Compare models, algorithms, solution templates | 📝 (+ small JumpStart listing) |
| 2.1.4 | Custom vs managed service vs pre-trained vs FM | 📝 |
| 2.1.5 | RAG architecture patterns | 📝 |
| 2.1.7 | AI model performance vs latency vs cost | ✅ |
| 2.1.8 | AWS AI services (Textract, Rekognition, Comprehend, Transcribe) | ✅ |

Skill 2.1.6 (ML performance vs training time vs cost) is in notebook 2.2a.

## Key points for the exam

- **Start with the least effort that works:** AI service → prompt an FM → RAG → fine-tune → train your own model.
- **Tabular data + need to explain decisions** → XGBoost / Linear Learner, not an FM.
- **Model knows too little about *your facts*** → RAG. **Model has the facts but the wrong style/format/behaviour** → fine-tuning.
- **Bigger FM = better reasoning, but slower and more expensive.** Pick the smallest model that passes your evaluation.
- **Inference profiles** route a request across Regions (cross-Region inference). Many newer models can only be called through one.
- **Distillation** = a big "teacher" model creates training data for a small "student" model: near-teacher quality at student price.
- **Continued pre-training** uses *unlabeled* text; **fine-tuning** uses *labeled* prompt/answer pairs.
- AI services need **no training data and no ML skills**: Textract (documents), Rekognition (images/video),
  Comprehend (text), Transcribe (speech to text), Translate, Polly (text to speech), Lex (chatbots).

| Need | Choose |
|---|---|
| Lowest latency and cost, simple task | Small FM (Nova Micro) |
| Complex reasoning, long documents | Large FM (Nova Pro) |
| Answers from private, changing documents | RAG (Bedrock Knowledge Bases) |
| Fixed tone, format or domain vocabulary | Fine-tuning |

## Setup

In [ ]:
import sys; sys.path.append("..")
import json, time
import pandas as pd
from common import utils as U, catchup as C
print(U.REGION, U.bucket())

Catch-up: make sure the raw SafeDrive files are in S3 (returns at once if they already are).

In [ ]:
RAW = C.ensure_raw_data()
bedrock, brt = U.client("bedrock"), U.client("bedrock-runtime")
print("raw data at", U.s3_uri(RAW))

## 2.1.1 Select a Bedrock foundation model

- Check four things: **modality** (text, image, embedding), **quality** on your task, **latency**, **price**.
- Also check: context window size, Region availability, whether it can be fine-tuned, and language support.
- Always test candidates with *your own* prompts. Public leaderboards do not know your data.

List the text models in this Region. Look at `inferenceTypesSupported`: `ON_DEMAND` means you can call the
model ID directly; `INFERENCE_PROFILE` means you must call it through an inference profile.

In [ ]:
models = bedrock.list_foundation_models(byOutputModality="TEXT")["modelSummaries"]
fm = pd.DataFrame(models)[["modelId", "providerName", "inputModalities",
                           "inferenceTypesSupported", "customizationsSupported"]]
print(len(fm), "text models")
fm.head(15)

> 🖥️ **Console check:** *Amazon Bedrock → Model catalog*.
> Filter by provider and modality and open one model card: it shows the model ID, supported use cases and the Regions where it runs. This is the same list in a form you can browse.

Which models can be customized? Filter by customization type. This matters for skill 2.1.2.

In [ ]:
ft = bedrock.list_foundation_models(byCustomizationType="FINE_TUNING")["modelSummaries"]
pd.DataFrame(ft)[["modelId", "customizationsSupported"]]

List the system-defined **inference profiles**. The ID starts with a geography (`us.`, `eu.`, `apac.`,
`global.`). `U.TEXT_MODEL` and friends are IDs from this list.

In [ ]:
profiles = bedrock.list_inference_profiles(typeEquals="SYSTEM_DEFINED", maxResults=100)
ip = pd.DataFrame(profiles["inferenceProfileSummaries"])[["inferenceProfileId", "status", "type"]]
print("we use:", U.SMALL_MODEL, "|", U.TEXT_MODEL, "|", U.LARGE_MODEL)
ip[ip["inferenceProfileId"].str.contains("nova")]

## 2.1.7 Performance vs latency vs cost (same prompt, three models)

- You pay per **input token** and per **output token**. Output tokens cost more.
- Larger models are slower and cost more per token. They help only if the task needs them.
- Ways to cut cost and latency: smaller model, shorter prompt, `maxTokens` limit, prompt caching,
  batch inference (about half price, not real time), latency-optimized inference, provisioned throughput.

The task: read one adjuster note and decide if the claim needs a fraud review. First load a note.

In [ ]:
notes = U.read_jsonl(f"{RAW}/adjuster_notes.jsonl")
note = next(n["note"] for n in notes if "SIU" in n["note"])
PROMPT = ("You are a motor insurance claims assistant. Read the adjuster note and answer in one line: "
          "REVIEW or ROUTINE, then a 10-word reason.\n\nNote: " + note)
print(PROMPT)

A helper that calls the Converse API and returns the answer with latency and token counts.
`usage` and `metrics` come back with every Converse response.

In [ ]:
def ask(model_id, prompt):
    start = time.time()
    r = brt.converse(modelId=model_id, messages=[{"role": "user", "content": [{"text": prompt}]}],
                     inferenceConfig={"maxTokens": 100, "temperature": 0})
    return {"model": model_id, "wall_s": round(time.time() - start, 2),
            "latency_ms": r["metrics"]["latencyMs"],
            "in_tokens": r["usage"]["inputTokens"], "out_tokens": r["usage"]["outputTokens"],
            "answer": r["output"]["message"]["content"][0]["text"].strip()}

ask(U.SMALL_MODEL, PROMPT)

Approximate on-demand prices in USD per 1,000 tokens (input, output). 💰 Prices change: check the
Amazon Bedrock pricing page before you rely on these numbers.

In [ ]:
PRICE = {"micro": (0.000035, 0.00014), "lite": (0.00006, 0.00024), "pro": (0.0008, 0.0032)}

def cost(row):
    p_in, p_out = next((v for k, v in PRICE.items() if k in row["model"]), (0, 0))
    return row["in_tokens"] / 1000 * p_in + row["out_tokens"] / 1000 * p_out

print(PRICE)

Run the same prompt on the three models. Compare latency, tokens, cost for 1 million such calls, and
whether the answers actually differ.

In [ ]:
rows = [ask(m, PROMPT) for m in (U.SMALL_MODEL, U.TEXT_MODEL, U.LARGE_MODEL)]
cmp = pd.DataFrame(rows)
cmp["usd_per_1M_calls"] = [round(cost(r) * 1_000_000, 2) for r in rows]
pd.set_option("display.max_colwidth", 120)
cmp

One call is noisy. Repeat 5 times per model and look at the median latency.

In [ ]:
lat = {m: sorted(ask(m, PROMPT)["latency_ms"] for _ in range(5))[2]
       for m in (U.SMALL_MODEL, U.TEXT_MODEL, U.LARGE_MODEL)}
pd.Series(lat, name="median_latency_ms")

**How to read it:** if all three give the same correct answer, the smallest model wins (cheapest and
fastest). Move up only when the small model fails your evaluation set (built in notebook 2.3b).

## 2.1.2 Fine-tuning strategies for foundation models 📝

| Approach | Changes model weights? | Data you need | Use when | Cost / effort |
|---|---|---|---|---|
| Prompt engineering | No | A few examples | First thing to try | Lowest |
| RAG | No | Your documents | Facts are private or change often; need citations | Low to medium |
| Fine-tuning | Yes | Labeled prompt → answer pairs (hundreds or more) | Fixed style, format, tone, narrow task | Medium to high |
| Continued pre-training | Yes | Large **unlabeled** domain text | Model must learn domain language (legal, medical) | High |
| Distillation | Yes (small student) | Prompts only; teacher writes the answers | Want big-model quality at small-model price and speed | Medium |

- On Bedrock all three are **model customization jobs**. To serve a custom model you use **Provisioned
  Throughput** or, for some models, on-demand custom model deployment.
- **PEFT / LoRA**: train small adapter layers instead of all weights. Cheaper, and less forgetting.
- On SageMaker AI you fine-tune open models from **JumpStart** or with your own training script.
- Fine-tuning does **not** keep the model up to date with new facts. That is the job of RAG.
- Hands-on: notebook 2.2b (skill 2.2.8).

**📘 Concept: Choosing between prompting, RAG and model customization**

Ask what is actually wrong with the model's answers, then pick the cheapest fix that addresses that cause.

| What is wrong | Fix | Why |
|---|---|---|
| The instructions are unclear, the format varies | Prompt engineering (role, rules, few-shot examples) | Nothing to train; change it in minutes |
| The model does not know your facts, or the facts change | RAG | Facts are fetched at query time and can be cited; update documents, not weights |
| It knows the facts but the style, tone or task behaviour is off, and examples in the prompt are not enough | Fine-tuning with labeled pairs | Behaviour is learned into the weights; prompts get shorter |
| It does not understand the domain's language at all, and you only have raw text | Continued pre-training | Learns vocabulary from unlabeled text |
| A large model does the task well but is too slow or expensive | Distillation | A small student model learns from the large teacher's answers |

- These combine: a fine-tuned model can still use RAG, and both still need a good prompt.
- Customization adds training cost, a custom model to host, and re-training when the base model changes. RAG adds a retrieval system to run. Prompting adds neither.
- Exam cues: "data changes daily", "must cite sources" → RAG. "consistent brand tone", "specific output format", "labeled examples" → fine-tuning. "large unlabeled corpus" → continued pre-training. "lower latency and cost with similar quality" → distillation. "least effort", "quickest" → prompt engineering.

## 2.1.3 Compare models, algorithms and solution templates 📝

**SageMaker AI built-in algorithms: cheat sheet**

| Problem | Algorithm | Remember |
|---|---|---|
| Tabular classification / regression | **XGBoost**, LightGBM, CatBoost, TabTransformer, AutoGluon-Tabular | XGBoost is the default answer for tabular data |
| Linear problems, very large sparse data | **Linear Learner** | Fast, easy to explain |
| Recommendations, sparse click data | **Factorization Machines** | Sparse, high-dimensional input |
| Classification / regression by neighbours | **k-NN** | Simple, index-based |
| Clustering | **K-Means** | Unsupervised, you choose k |
| Dimensionality reduction | **PCA** | Unsupervised |
| Anomaly detection | **Random Cut Forest (RCF)** | Unsupervised anomaly score |
| Suspicious IP addresses | **IP Insights** | Learns (entity, IP) pairs |
| Time-series forecasting | **DeepAR** | Many related series |
| Text classification, word vectors | **BlazingText** | Word2vec and fast text classifier |
| Topic modeling | **LDA**, **NTM** | Unsupervised topics |
| Translation-style sequence tasks | **Seq2Seq** | Needs a GPU |
| Embeddings for pairs of objects | **Object2Vec** | General-purpose embeddings |
| Images | **Image Classification**, **Object Detection**, **Semantic Segmentation** | Need a GPU for training |

| Business need | Lean towards |
|---|---|
| Interpretability (regulator asks "why?") | Linear Learner, shallow XGBoost + SHAP (SageMaker Clarify) |
| Lowest latency | Small tree model or linear model; small FM |
| Little labeled data | Pre-trained model + fine-tuning (JumpStart), or an FM with few-shot prompts |
| No ML team | AI services, SageMaker Autopilot / Canvas |

- **SageMaker JumpStart** = model hub with pre-trained models and solution templates you can deploy or fine-tune.
- **Autopilot (AutoML)** tries algorithms and settings for you and gives a leaderboard.

A small JumpStart listing: tabular classification models you could train instead of writing your own.
The call reads a public catalog; nothing is deployed.

In [ ]:
import sagemaker
from sagemaker.jumpstart.notebook_utils import list_jumpstart_models
sm_session = sagemaker.Session(boto_session=U.session, default_bucket=U.bucket())
js = list_jumpstart_models(filter="task == classification", region=U.REGION, sagemaker_session=sm_session)
print(len(js), "tabular classification models, for example:")
js[:10]

## 2.1.4 Custom vs managed AI service vs pre-trained model vs FM 📝

| Option | Effort | Control | Data needed | Good fit |
|---|---|---|---|---|
| **AWS AI service** (Textract, Comprehend ...) | Lowest | Low | None | Common task: OCR, sentiment, speech to text |
| **Foundation model** (Bedrock) | Low | Medium (prompt, RAG, fine-tune) | None to little | Language tasks, summaries, chat, extraction |
| **Pre-trained model** (JumpStart) + fine-tune | Medium | High | Some labeled data | Need to host the model yourself or tune it deeply |
| **Custom model** (SageMaker training) | Highest | Full | Lots of labeled data | Unique problem, tabular data, strict latency or cost |

SafeDrive scenarios:

| Problem | Best choice | Why |
|---|---|---|
| Fraud score from 27 numeric features | Custom XGBoost on SageMaker | Tabular, labeled history, must be explainable |
| Read fields from scanned claim forms | Amazon Textract | Standard OCR task, no training |
| Answer policy questions with sources | Bedrock FM + RAG | Private documents that change |
| Summarize adjuster notes | Bedrock FM with a prompt | General language skill is enough |
| Classify photos into dent / scratch / broken glass | Rekognition Custom Labels or a fine-tuned JumpStart vision model | Company-specific labels |

- Think about **total cost**: an FM has no training cost but a per-token cost forever; a custom model
  has training and hosting cost but can be very cheap per prediction.
- Also weigh data privacy, the skills of the team, and time to market.

## 2.1.5 RAG architecture patterns 📝

| Pattern | How it works | Choose when |
|---|---|---|
| **Naive (vector) RAG** | Embed the question, fetch top-k chunks, generate | Simple Q&A over clean documents |
| **Hybrid search** | Vector search + keyword (BM25) search, merged | Exact terms matter: IDs, product codes, names |
| **Rerank** | Fetch many (for example 20), a reranker model keeps the best few | Top-k has the right chunk but in a low position |
| **Agentic RAG** | An agent decides what to search, can search many times and call tools | Multi-step questions, several sources |
| **GraphRAG** | Entities and relations in a graph (Neptune Analytics) + vectors | Questions that link facts across documents |
| **Structured / SQL RAG** | The model writes SQL against a database (Redshift, Athena) | Answers live in tables: counts, sums, filters |
| **Multimodal RAG** | Parse or embed images, tables and scans | PDFs with charts, scanned forms, photos |

- Add **metadata filters** (for example `year = 2026`) to cut wrong-context answers.
- **Query rewriting / decomposition** helps vague or multi-part questions.
- Bedrock Knowledge Bases supports all of the above as managed options.
- Hands-on: retrieval tuning in 2.2b, Knowledge Base and patterns in 3.1b and 3.2b.

## 2.1.8 AWS AI services on claim files

- One API call each, pay per use, no model to train or host.
- Each service has a **custom** option when the generic model is not enough: Comprehend custom
  classifier, Rekognition Custom Labels, Transcribe custom vocabulary, Textract custom queries.

### Amazon Textract: read the scanned claim form
`detect_document_text` returns plain lines of text (OCR). Look at the lines it found.

In [ ]:
textract = U.client("textract")
form = {"S3Object": {"Bucket": U.bucket(), "Name": f"{RAW}/images/claim_form.png"}}
ocr = textract.detect_document_text(Document=form)
lines = [b["Text"] for b in ocr["Blocks"] if b["BlockType"] == "LINE"]
lines

`analyze_document` with **QUERIES** lets you ask for a field in plain English. `FORMS` and `TABLES`
are the other feature types. Look at each answer and its confidence.

In [ ]:
questions = ["What is the claim ID?", "What is the policy number?", "What is the claimed amount?"]
doc = textract.analyze_document(Document=form, FeatureTypes=["QUERIES"],
                                QueriesConfig={"Queries": [{"Text": q} for q in questions]})
answers = [(b["Text"], round(b["Confidence"], 1)) for b in doc["Blocks"] if b["BlockType"] == "QUERY_RESULT"]
answers

### Amazon Rekognition: labels on a damage photo
`detect_labels` finds objects and scenes. Our photos are simple drawings, so expect generic labels.
For "dent vs scratch" you would train **Rekognition Custom Labels**.

In [ ]:
rek = U.client("rekognition")
manifest = U.read_jsonl(f"{RAW}/images/manifest.jsonl")
photo = {"S3Object": {"Bucket": U.bucket(), "Name": f"{RAW}/images/{manifest[0]['file']}"}}
labels = rek.detect_labels(Image=photo, MaxLabels=8, MinConfidence=50)["Labels"]
print("true metadata:", manifest[0])
pd.DataFrame(labels)[["Name", "Confidence"]]

### Amazon Comprehend: sentiment, entities and PII in adjuster notes
First the sentiment of a note from an upset customer.

In [ ]:
comprehend = U.client("comprehend")
upset = next(n["note"] for n in notes if "upset" in n["note"])
s = comprehend.detect_sentiment(Text=upset, LanguageCode="en")
print(upset)
print(s["Sentiment"], {k: round(v, 2) for k, v in s["SentimentScore"].items()})

Entities (people, dates, quantities ...) in the same note.

In [ ]:
ents = comprehend.detect_entities(Text=upset, LanguageCode="en")["Entities"]
pd.DataFrame(ents)[["Text", "Type", "Score"]]

PII detection returns offsets, so you can mask the text before it goes into a prompt or a training set.

In [ ]:
pii = comprehend.detect_pii_entities(Text=upset, LanguageCode="en")["Entities"]
[(p["Type"], upset[p["BeginOffset"]:p["EndOffset"]], round(p["Score"], 2)) for p in pii]

### Amazon Transcribe: speech to text for a recorded call
Transcribe works as an asynchronous **job** (there is also a streaming API). Start the job.

In [ ]:
transcribe = U.client("transcribe")
JOB = f"safedrive-call-{int(time.time())}"
OUT_KEY = f"{U.PREFIX}/transcribe/call_sample.json"
transcribe.start_transcription_job(
    TranscriptionJobName=JOB, LanguageCode="en-US", MediaFormat="mp3",
    Media={"MediaFileUri": U.s3_uri(f"{RAW}/audio/call_sample.mp3")},
    OutputBucketName=U.bucket(), OutputKey=OUT_KEY, Tags=U.TAGS)
print("started", JOB)

Wait for the job (usually under a minute for a short file).

In [ ]:
U.wait_until(lambda: transcribe.get_transcription_job(TranscriptionJobName=JOB)
             ["TranscriptionJob"]["TranscriptionJobStatus"],
             ok=("COMPLETED",), bad=("FAILED",), delay=10, label="transcribe job")

> 🖥️ **Console check:** *Amazon Transcribe → Transcription jobs → the `safedrive-call-...` job*.
> Look at the transcription preview and the output location. The same page is where you would turn on options such as speaker identification or PII redaction for a new job.

Read the transcript JSON from S3.

In [ ]:
result = json.loads(U.read_bytes(OUT_KEY))
transcript = result["results"]["transcripts"][0]["transcript"]
print(transcript)

Chain two choices together: the transcript goes to the small FM for a one-line summary.
This is a common pattern: AI service for perception, FM for language.

In [ ]:
print(U.converse("Summarize this insurance call in one sentence:\n" + transcript, model_id=U.SMALL_MODEL))

## Exam traps

- "Company documents change every week" → RAG, **not** fine-tuning.
- "Unlabeled domain text" → continued pre-training. "Labeled prompt/answer pairs" → fine-tuning.
- "Tabular data, must explain each decision" → XGBoost or Linear Learner with SHAP, not an FM.
- "Extract fields from scanned forms" → Textract (not Rekognition, not Comprehend).
- "Least operational overhead" almost always points to a managed AI service or Bedrock, not a custom model.

## Quiz

**Q1.** A team needs answers from a product manual that is updated daily. Answers must cite the page.
Which approach has the least effort?
A) Fine-tune an FM every night  B) RAG with a Bedrock Knowledge Base  C) Continued pre-training  D) Train a Seq2Seq model

<details><summary>Answer</summary>B. RAG reads the current documents at query time and can return citations. No retraining.</details>

**Q2.** A call center wants to use a large FM's answer quality but needs much lower cost and latency for one
narrow task. They have many example prompts but no labeled answers. What fits best?
A) Model distillation  B) Continued pre-training  C) Provisioned Throughput on the large model  D) Raise the temperature

<details><summary>Answer</summary>A. Distillation lets the large teacher write the answers and trains a small student model on them.</details>

**Q3.** A bank must predict loan default from 40 numeric columns and explain every decision to a regulator.
Which choice is best?
A) Bedrock FM with few-shot prompts  B) SageMaker XGBoost with SageMaker Clarify  C) Amazon Comprehend  D) DeepAR

<details><summary>Answer</summary>B. Tabular supervised problem; tree model plus SHAP values gives per-prediction explanations.</details>

**Q4.** Users search a knowledge base for exact part numbers such as `XK-4471`. Vector search often returns
similar but wrong parts. What should you add first?
A) A larger embedding dimension  B) Hybrid search (keyword + vector)  C) A larger FM  D) Higher top-k only

<details><summary>Answer</summary>B. Keyword matching catches exact tokens that embeddings blur together.</details>

**Q5.** A company wants text from recorded phone calls, then the customer's sentiment, with no ML team.
Which services?
A) Polly then Lex  B) Transcribe then Comprehend  C) Textract then Rekognition  D) SageMaker BlazingText only

<details><summary>Answer</summary>B. Transcribe turns speech into text; Comprehend detects sentiment. Both are managed AI services.</details>

## Save state
This notebook is about choices, so it only records that it ran.

In [ ]:
state = U.save_state(nb_2_1_done=True)
print(sorted(state))

## Cleanup
The AI service calls and Bedrock calls are pay-per-request and leave nothing running. Only the
Transcribe job record and its output file remain. Nothing is kept for later notebooks.

### Before you delete: what later notebooks use

**Nothing from this notebook is needed later.** It is safe to delete everything it created.

In [ ]:
U.safe(transcribe.delete_transcription_job, TranscriptionJobName=JOB)

In [ ]:
U.safe(U.client("s3").delete_object, Bucket=U.bucket(), Key=OUT_KEY)